### Imorts and loads unique coordinates

In [1]:
import geopandas as gpd
import osmnx as ox
from shapely.geometry import Point, MultiPolygon, mapping, box
import json
import numpy as np
import pandas as pd
import shapely

In [2]:
points = gpd.read_parquet("./data/clean.parquet")[["geo_lat", "geo_lon", "geometry"]]
points = points.drop_duplicates(subset=["geo_lat", "geo_lon"])
points = points.to_crs(32637)
points.head(), points.shape

(      geo_lat    geo_lon                        geometry
 20  55.738876  37.825370  POINT (426249.603 6177642.656)
 29  55.798921  37.738090  POINT (420891.649 6184421.021)
 34  55.655307  37.614605  POINT (412831.601 6168587.463)
 44  55.701330  37.507412  POINT (406197.791 6173848.674)
 56  55.730785  37.631424  POINT (414055.506 6176965.665),
 (46639, 3))

### Metro

In [3]:
stations = ox.features_from_place("Moscow, Russia", tags={"station": "subway"})

In [4]:
stations["railway"].value_counts(dropna=False)
stations.geom_type.value_counts()      # должны быть в основном Point
stations["name"].nunique()

242

In [5]:
msk = gpd.read_file("./data/moscow_city.geojson")         
area = (msk.to_crs(32637)          
           .buffer(5000)           
           .union_all()            
           .simplify(200))         

area_wgs = gpd.GeoSeries([area], crs=32637).to_crs(4326).iloc[0]

geojson_data = {
    "type": "MultiPolygon",
    "coordinates": mapping(area_wgs)["coordinates"]
}

with open("./data/msk_area_mask.geojson", "w", encoding="utf-8") as file:
    json.dump(geojson_data, file)


In [6]:
stations = gpd.read_file("./data/osm/metro.geojson")

In [7]:
stations.info()

<class 'geopandas.geodataframe.GeoDataFrame'>
RangeIndex: 281 entries, 0 to 280
Data columns (total 5 columns):
 #   Column    Non-Null Count  Dtype   
---  ------    --------------  -----   
 0   element   281 non-null    str     
 1   name      281 non-null    str     
 2   railway   278 non-null    str     
 3   station   281 non-null    str     
 4   geometry  281 non-null    geometry
dtypes: geometry(1), str(4)
memory usage: 22.1 KB


In [8]:
metro = stations.reset_index()[["geometry", "name"]]
metro.describe()

,geometry,name
count,281,281
unique,281,244
top,POINT (37.66155 55.8871767),Деловой центр
freq,1,3


In [9]:
metro = metro[metro.geom_type == "Point"]
metro = metro.to_crs(32637)

metro_grouped = (
    metro
    .dissolve(
        by="name",
        aggfunc="first",
    )
    .reset_index()
)

metro_grouped["geometry"] = metro_grouped.geometry.centroid

metro_grouped.describe()

,name,geometry
count,244,244
unique,244,244
top,Авиамоторная,POINT (419634.07356757915 6179357.799152892)
freq,1,1


In [10]:
dists = gpd.sjoin_nearest(
    points, 
    metro_grouped, 
    how="left",
    distance_col="distance_metro",
)

dists = dists[["geo_lat", "geo_lon", "distance_metro"]]

dists.describe()

,geo_lat,geo_lon,distance_metro
count,46639.000000,46639.000000,46639.000000
mean,55.735213,37.588813,1572.642113
std,0.105207,0.149964,3336.719796
min,55.179939,36.836900,8.073499
25%,55.670130,37.500436,536.850577
50%,55.743035,37.591158,814.833734
75%,55.807979,37.687951,1254.156594
max,56.011032,37.950668,42645.877856


In [11]:
points = points.merge(
    dists, 
    on=["geo_lat", "geo_lon"],
    how="left"
)
points.head(), points.shape

(     geo_lat    geo_lon                        geometry  distance_metro
 0  55.738876  37.825370  POINT (426249.603 6177642.656)     1533.749990
 1  55.798921  37.738090  POINT (420891.649 6184421.021)      702.905952
 2  55.655307  37.614605  POINT (412831.601 6168587.463)      378.381579
 3  55.701330  37.507412  POINT (406197.791 6173848.674)      697.747350
 4  55.730785  37.631424  POINT (414055.506 6176965.665)      418.445216,
 (46639, 4))

### Rails

In [12]:
rail = gpd.read_file("./data/osm/rail.geojson")
rail.info()

<class 'geopandas.geodataframe.GeoDataFrame'>
RangeIndex: 544 entries, 0 to 543
Data columns (total 6 columns):
 #   Column    Non-Null Count  Dtype   
---  ------    --------------  -----   
 0   element   544 non-null    str     
 1   name      543 non-null    str     
 2   railway   544 non-null    str     
 3   station   279 non-null    str     
 4   landuse   2 non-null      str     
 5   geometry  544 non-null    geometry
dtypes: geometry(1), str(5)
memory usage: 44.6 KB


In [13]:
rail.describe()

,element,name,railway,station,landuse,geometry
count,544,543,544,279,2,544
unique,2,438,2,2,1,544
top,node,Кунцевская,station,subway,railway,POINT (37.6537207 55.7788343)
freq,542,4,376,278,2,1


In [14]:
rail = rail[rail.geom_type == "Point"]
rail = rail.to_crs(32637)

rail_grouped = (
    rail
    .dissolve(
        by="name",
        aggfunc="first",
    )
    .reset_index()
)

rail_grouped["geometry"] = rail_grouped.geometry.centroid

rail_grouped.describe()

,name,geometry,element,railway,station,landuse
count,437,437,437,437,242,0
unique,437,437,1,2,1,0
top,240 км,POINT (367014.3349236073 6140064.395899067),node,station,subway,NaN
freq,1,1,437,309,242,NaN


In [15]:
rail_dists = gpd.sjoin_nearest(
    points, 
    rail_grouped, 
    how="left",
    distance_col="distance_rail",
)

rail_dists = rail_dists[["geo_lat", "geo_lon", "distance_rail"]]

rail_dists.describe()

,geo_lat,geo_lon,distance_rail
count,46639.000000,46639.000000,46639.000000
mean,55.735213,37.588813,1041.894608
std,0.105207,0.149964,1535.622976
min,55.179939,36.836900,7.042217
25%,55.670130,37.500436,497.035158
50%,55.743035,37.591158,736.811224
75%,55.807979,37.687951,1058.668211
max,56.011032,37.950668,17869.219503


In [16]:
points = points.merge(
    rail_dists, 
    on=["geo_lat", "geo_lon"],
    how="left"
)
points.head(), points.shape

(     geo_lat    geo_lon                        geometry  distance_metro  \
 0  55.738876  37.825370  POINT (426249.603 6177642.656)     1533.749990   
 1  55.798921  37.738090  POINT (420891.649 6184421.021)      702.905952   
 2  55.655307  37.614605  POINT (412831.601 6168587.463)      378.381579   
 3  55.701330  37.507412  POINT (406197.791 6173848.674)      697.747350   
 4  55.730785  37.631424  POINT (414055.506 6176965.665)      418.445216   
 
    distance_rail  
 0    1134.700201  
 1     702.905952  
 2     413.705421  
 3     697.747350  
 4     418.445216  ,
 (46639, 5))

### Roads

In [17]:
roads = gpd.read_file("./data/osm/roads.geojson")

In [18]:
roads.describe()

,element,highway,name,railway,lanes,waterway,geometry
count,43479,43479,37498,11,39464,1,43479
unique,1,5,2412,1,8,1,43250
top,way,tertiary,Варшавское шоссе,razed,2,dam,"LINESTRING (37.6241214 55.7497242, 37.6238065 ..."
freq,43479,17379,358,11,15859,1,2


In [19]:
roads["lanes"] = roads["lanes"].astype("Int64")

roads["lanes"] = roads.groupby("highway")["lanes"].transform("median")
roads.info()

<class 'geopandas.geodataframe.GeoDataFrame'>
RangeIndex: 43479 entries, 0 to 43478
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype   
---  ------    --------------  -----   
 0   element   43479 non-null  str     
 1   highway   43479 non-null  str     
 2   name      37498 non-null  str     
 3   railway   11 non-null     str     
 4   lanes     43479 non-null  Float64 
 5   waterway  1 non-null      str     
 6   geometry  43479 non-null  geometry
dtypes: Float64(1), geometry(1), str(5)
memory usage: 4.1 MB


In [20]:
roads = roads[(roads.geom_type =="LineString") | (roads.geom_type ==  "MultiLineString")].explode()
roads = roads.to_crs(32637)

roads.info()

<class 'geopandas.geodataframe.GeoDataFrame'>
RangeIndex: 43479 entries, 0 to 43478
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype   
---  ------    --------------  -----   
 0   element   43479 non-null  str     
 1   highway   43479 non-null  str     
 2   name      37498 non-null  str     
 3   railway   11 non-null     str     
 4   lanes     43479 non-null  Float64 
 5   waterway  1 non-null      str     
 6   geometry  43479 non-null  geometry
dtypes: Float64(1), geometry(1), str(5)
memory usage: 4.1 MB


In [21]:
data = {"motorway": 5, "trunk": 4, "primary": 3, "secondary": 2, "tertiary": 1}

weights = gpd.GeoDataFrame(
    pd.DataFrame(list(data.items()), columns=["highway", "weight"])
)

weights.head()

,highway,weight
0,motorway,5
1,trunk,4
2,primary,3
3,secondary,2
4,tertiary,1


In [22]:
roads = roads.merge(
    weights,
    on="highway",
    how="left",
)
print(roads.head()) 
roads.info()

  element    highway                name railway  lanes waterway  \
0     way   tertiary  Барвихинская улица     NaN    2.0      NaN   
1     way   tertiary  Барвихинская улица     NaN    2.0      NaN   
2     way   tertiary     улица Толбухина     NaN    2.0      NaN   
3     way  secondary   Беловежская улица     NaN    3.0      NaN   
4     way   tertiary   Запорожская улица     NaN    2.0      NaN   

                                            geometry  weight  
0  LINESTRING (399464.082 6175612.154, 399437.261...       1  
1  LINESTRING (399393.879 6175607.518, 399310.244...       1  
2  LINESTRING (399452.574 6176530.405, 399459.137...       1  
3  LINESTRING (399501.403 6174231.613, 399492.052...       2  
4  LINESTRING (399768.116 6176407.541, 399775.467...       1  
<class 'geopandas.geodataframe.GeoDataFrame'>
RangeIndex: 43479 entries, 0 to 43478
Data columns (total 8 columns):
 #   Column    Non-Null Count  Dtype   
---  ------    --------------  -----   
 0   element   43

In [23]:
roads["load"] = roads["lanes"] * roads["weight"]
roads.shape

(43479, 9)

In [24]:
major_roads = roads[(roads["highway"] == 'motorway') | (roads["highway"] == 'trunk')]

roads_dists = gpd.sjoin_nearest(
    points, 
    major_roads[["geometry"]], 
    how="left",
    distance_col="d",
)

roads_dists.describe(), roads_dists.duplicated().sum()

(            geo_lat       geo_lon  distance_metro  distance_rail  \
 count  50825.000000  50825.000000    50825.000000   50825.000000   
 mean      55.734298     37.590405     1555.430903    1050.484282   
 std        0.104378      0.148935     3298.706215    1557.785594   
 min       55.179939     36.836900        8.073499       7.042217   
 25%       55.673044     37.501581      531.119669     495.000172   
 50%       55.744263     37.592706      805.768318     735.516460   
 75%       55.805422     37.688964     1229.753911    1058.237594   
 max       56.011032     37.950668    42645.877856   17869.219503   
 
         index_right             d  
 count  50825.000000  50825.000000  
 mean    9634.453596   5532.255614  
 std     9487.662792   3877.061179  
 min      119.000000      0.173562  
 25%     3190.000000   2290.882789  
 50%     3266.000000   4577.836046  
 75%    15853.000000   8565.772307  
 max    42958.000000  14643.037450  ,
 np.int64(0))

In [25]:
points["distance_major_road_m"] = roads_dists.groupby(level=0)["d"].first()
points.shape

(46639, 6)

In [26]:
buf = points.buffer(300)
pairs = gpd.sjoin(gpd.GeoDataFrame(geometry=buf), roads[["geometry", "load"]],
                  predicate="intersects")

clipped_len = shapely.intersection(
    buf.loc[pairs.index].values,
    roads.geometry.loc[pairs["index_right"]].values,
)                           


In [27]:
lengths_inside = shapely.length(clipped_len)


In [28]:
pairs["w_len"] = lengths_inside * pairs["load"].values
points["road_load_300"] = pairs.groupby(level=0)["w_len"].sum().reindex(points.index, fill_value=0)
points.sample(5)

,geo_lat,geo_lon,geometry,distance_metro,distance_rail,distance_major_road_m,road_load_300
37701,55.737668,37.573803,POINT (410453.015 6177804.533),897.678633,840.424431,11362.271123,6637.528375
19930,55.979621,37.185573,POINT (386785.681 6205297.459),17321.140812,727.570189,3949.554898,15083.105333
43140,55.720337,37.582421,POINT (410954.642 6175864.853),772.196528,772.196528,11126.190216,8405.798538
3734,55.590144,37.597288,POINT (411595.536 6161358.123),574.298254,574.298254,1505.034160,18217.901192
9056,55.804737,37.399675,POINT (399693.351 6185506.445),242.773837,242.773837,777.706761,1179.211651


In [29]:
buf = points.buffer(1000)
pairs = gpd.sjoin(gpd.GeoDataFrame(geometry=buf), roads[["geometry", "load"]],
                  predicate="intersects")

clipped_len = shapely.intersection(
    buf.loc[pairs.index].values,
    roads.geometry.loc[pairs["index_right"]].values,
)                           
lengths_inside = shapely.length(clipped_len)
pairs["w_len"] = lengths_inside * pairs["load"].values
points["road_load_1000"] = pairs.groupby(level=0)["w_len"].sum().reindex(points.index, fill_value=0)
points.sample(5)


,geo_lat,geo_lon,geometry,distance_metro,distance_rail,distance_major_road_m,road_load_300,road_load_1000
42293,55.848990,37.476205,POINT (404598.148 6190322.761),670.413544,670.413544,3839.082408,11002.023054,48646.783335
12439,55.837715,37.664043,POINT (416333.27 6188825.119),1799.981956,380.774151,6270.204504,6930.074436,88419.738559
14436,55.830836,37.646694,POINT (415231.844 6188080.74),1152.384986,1152.384986,7263.803180,11582.158109,89778.638451
30524,55.598653,37.604845,POINT (412090.806 6162295.359),493.710259,493.710259,2504.426103,17355.293338,85256.345475
23452,55.841705,37.536266,POINT (408341.006 6189430.925),2244.517742,768.894091,6019.666572,0.0,46176.732082


In [30]:
estates = gpd.read_parquet("./data/clean.parquet")
estates = estates.merge(
    points[["geo_lat", "geo_lon", "distance_metro", "distance_rail", "road_load_300", "road_load_1000"]],
    on=("geo_lat", "geo_lon"),
    how="left",
)
estates.info()

<class 'geopandas.geodataframe.GeoDataFrame'>
RangeIndex: 375638 entries, 0 to 375637
Data columns (total 23 columns):
 #   Column          Non-Null Count   Dtype         
---  ------          --------------   -----         
 0   price           375638 non-null  int64         
 1   date            375638 non-null  str           
 2   time            375638 non-null  str           
 3   geo_lat         375638 non-null  float64       
 4   geo_lon         375638 non-null  float64       
 5   region          375638 non-null  int64         
 6   building_type   375638 non-null  int64         
 7   level           375638 non-null  int64         
 8   levels          375638 non-null  int64         
 9   rooms           375638 non-null  int64         
 10  area            375638 non-null  float64       
 11  kitchen_area    375638 non-null  float64       
 12  object_type     375638 non-null  int64         
 13  dt              375638 non-null  datetime64[us]
 14  geometry        375638 non-n

In [31]:
estates.to_parquet("./data/clean_with_transport.parquet")

In [32]:
# estates['distance_metro'].isna().sum()